# Lección 9.1 — La neurona artificial: suma ponderada + decisión

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente), Lección 5.2 (regresión logística), Lección 2.3 (escalado)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué hace una neurona artificial: **sumar evidencias con pesos y decidir**.
- Calcular **a mano** la salida de una neurona (suma ponderada + sesgo + activación).
- Describir cómo **aprende** un perceptrón y por qué **no puede** aprender el XOR.
- Reconocer que una neurona con sigmoide **es** una regresión logística, y entrenar una en PyTorch.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos. **PyTorch** (`torch`) es la librería de redes neuronales que usaremos en todo el módulo; en Google Colab ya viene instalada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"   # en Colab sin GPU será "cpu": no pasa nada
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: ¿voy a la playa?

Es sábado por la mañana y decides si vas a la playa. Sin darte cuenta, **sumas evidencias**:

| Evidencia | ¿Se cumple? (1 = sí) | Cuánto te importa (peso) |
|---|---|---|
| ☀️ Hace sol | 1 | **+3** |
| 🧑‍🤝‍🧑 Van tus amigos | 0 | **+2** |
| 💸 Estás corto de dinero | 1 | **−1** |

Haces la cuenta: `3·1 + 2·0 + (−1)·1 = 2`. Y luego **decides**: si la suma pasa de cierto **umbral** (digamos 2.5), vas; si no, te quedas.

Eso es, literalmente, una **neurona artificial**:
1. Recibe **entradas** (las evidencias).
2. Multiplica cada una por un **peso** (cuánto importa, y si empuja a favor o en contra).
3. Suma todo y le añade un **sesgo** (*bias*): qué tan "fácil de convencer" es.
4. Pasa el resultado por una **función de activación**, que toma la decisión.

```
 sol ───(×3)───┐
 amigos ─(×2)──┼──► Σ + sesgo ──► activación ──► ¿voy? (sí / no)
 sin $ ──(×−1)─┘
```

### ✋ Ejercicio 1 — Sé la neurona
Con los pesos de la tabla y un umbral de 2.5, ¿vas a la playa en estos tres sábados?

| Sábado | ☀️ Sol | 🧑‍🤝‍🧑 Amigos | 💸 Sin dinero |
|---|---|---|---|
| A | 1 | 0 | 1 |
| B | 0 | 1 | 0 |
| C | 1 | 1 | 1 |

✍️ *Tus respuestas:* A → ___ · B → ___ · C → ___

<details><summary><b>👀 Ver respuesta</b></summary>

- **A:** 3 + 0 − 1 = **2** → no pasa 2.5 → ❌ te quedas (es el ejemplo de arriba).
- **B:** 0 + 2 + 0 = **2** → ❌ te quedas. Tus amigos solos no bastan.
- **C:** 3 + 2 − 1 = **4** → ✅ vas.

Fíjate: el peso negativo del dinero **resta** evidencia. Y el umbral decide qué tan exigente eres: con umbral 1.5, los tres sábados irías.
</details>

---
## 2. 📖 El concepto: la neurona en dos pasos

Toda neurona hace siempre lo mismo:

**Paso 1 — Suma ponderada:** `z = w₁·x₁ + w₂·x₂ + … + b`

**Paso 2 — Activación (decisión):** `salida = f(z)`

| Término | Qué es | En la analogía |
|---|---|---|
| **Entradas** `x` | Las características del ejemplo | Sol, amigos, dinero |
| **Pesos** `w` | Cuánto importa cada entrada (y su signo) | +3, +2, −1 |
| **Sesgo** `b` | Un desplazamiento fijo; equivale a mover el umbral | −2.5 ("necesito al menos 2.5 de evidencia") |
| **Activación** `f` | Convierte la suma en una decisión o probabilidad | "si pasa de 0, voy" |

> 💡 El truco del sesgo: "¿la suma pasa de 2.5?" es lo mismo que "¿la suma **− 2.5** pasa de 0?". Por eso en vez de umbral se usa un sesgo `b = −2.5` y la activación compara siempre contra 0.

Los **pesos y el sesgo son los parámetros**: las "perillas" de la Lección 3.1. Aprender = encontrar buenos valores para ellos.

Así se ve en código, a mano con `numpy`:

In [ ]:
def neurona_escalon(x, w, b):
    z = np.dot(w, x) + b          # paso 1: suma ponderada + sesgo
    return 1 if z > 0 else 0      # paso 2: activación "escalón" (sí/no)

w = np.array([3, 2, -1])          # pesos: sol, amigos, sin dinero
b = -2.5                          # sesgo (umbral 2.5)

for nombre, x in {"A": [1, 0, 1], "B": [0, 1, 0], "C": [1, 1, 1]}.items():
    print(f"Sábado {nombre}: z = {np.dot(w, x) + b:+.1f} → {'✅ voy' if neurona_escalon(x, w, b) else '❌ me quedo'}")

### ✏️ Ejercicio 2 — Cambia la personalidad de la neurona
1. Cambia el sesgo a `b = -1.5`. ¿Qué sábados cambian de decisión?
2. Vuelve a `b = -2.5` y ahora cambia el peso de los amigos de `2` a `4` (eres muy sociable). ¿Qué pasa con el sábado B?

✍️ *Mis predicciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con `b = -1.5`, A y B pasan a `z = +0.5` → ✅ vas los tres sábados. Un sesgo menos negativo = una neurona **más fácil de convencer**.
2. Con peso 4 para los amigos, el sábado B da `z = 4 − 2.5 = +1.5` → ✅ vas. Un peso grande significa que esa entrada **sola** puede decidir.

Los pesos dicen **qué** importa; el sesgo dice **cuánta** evidencia hace falta en total.
</details>

### 📖 Del "sí/no" a la probabilidad: la sigmoide
El escalón es tajante: con `z = 0.01` dice "sí" con la misma seguridad que con `z = 100`. Casi siempre preferimos una respuesta **gradual**, una probabilidad. Para eso se usa la **sigmoide** (la misma de la Lección 5.2):

`sigmoide(z) = 1 / (1 + e^(−z))`

- `z` muy negativo → casi 0
- `z = 0` → exactamente 0.5
- `z` muy positivo → casi 1

In [ ]:
z = np.linspace(-8, 8, 200)
escalon = (z > 0).astype(float)
sigmoide = 1 / (1 + np.exp(-z))

plt.plot(z, escalon, label="escalón (sí/no)", lw=2)
plt.plot(z, sigmoide, label="sigmoide (probabilidad)", lw=2)
plt.axvline(0, color="gray", ls=":"); plt.xlabel("z = suma ponderada + sesgo"); plt.ylabel("salida")
plt.legend(); plt.grid(alpha=.3); plt.title("Dos funciones de activación"); plt.show()

### 🔮 Ejercicio 3 — Sin calculadora
Con la sigmoide como activación (y los pesos originales, `b = -2.5`), ordena los sábados A, B y C de **menor a mayor** probabilidad de ir. ¿Alguno da **exactamente** 0.5? ¿Alguno pasa de 0.8?

✍️ *Mi predicción:*

In [ ]:
def neurona_sigmoide(x, w, b):
    z = np.dot(w, x) + b
    return 1 / (1 + np.exp(-z))

for nombre, x in {"A": [1, 0, 1], "B": [0, 1, 0], "C": [1, 1, 1]}.items():
    print(f"Sábado {nombre}: z = {np.dot(w, x) + b:+.1f} → probabilidad de ir = {neurona_sigmoide(x, w, b):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

A y B empatan (`z = −0.5` → **0.38**) y C es el más alto (`z = +1.5` → **0.82**). Ninguno da 0.5 exacto (haría falta `z = 0`). Ahora la neurona no solo decide: dice **qué tan segura** está. Para decidir, basta con comparar la probabilidad contra 0.5.
</details>

---
## 3. 📖 Cómo aprende: el perceptrón (1958)

Hasta ahora **nosotros** elegimos los pesos. La gracia del ML es que la neurona los **aprenda de ejemplos**. El primer algoritmo para eso fue el **perceptrón** de Frank Rosenblatt, y su regla es casi de sentido común:

> Muestra un ejemplo. Si la neurona **acierta**, no toques nada. Si **se equivoca**, mueve cada peso un poquito **en la dirección que habría arreglado el error**.

En fórmula: `peso ← peso + tasa · (correcto − predicho) · entrada`

- Si dijo 0 y era 1 → `(1 − 0) = +1` → **sube** los pesos de las entradas encendidas.
- Si dijo 1 y era 0 → `(0 − 1) = −1` → **baja** esos pesos.
- Si acertó → `0` → no cambia nada.

Probémoslo con la puerta lógica **AND** ("las dos entradas encendidas"):

In [ ]:
X_logica = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y_AND = np.array([0, 0, 0, 1])

def entrenar_perceptron(X, y, epocas=10, tasa=0.1, verbose=True):
    w, b = np.zeros(X.shape[1]), 0.0                 # empezamos sin saber nada
    for epoca in range(1, epocas + 1):
        errores = 0
        for xi, yi in zip(X, y):
            pred = 1 if np.dot(w, xi) + b > 0 else 0
            w += tasa * (yi - pred) * xi                 # la regla del perceptrón
            b += tasa * (yi - pred)
            errores += int(pred != yi)
        if verbose:
            print(f"época {epoca:>2}: errores = {errores}   pesos = {np.round(w, 2)}   sesgo = {b:+.2f}")
    return w, b

w_and, b_and = entrenar_perceptron(X_logica, Y_AND)

### ✋ Ejercicio 4 — Lee el aprendizaje
1. ¿En qué época deja de equivocarse?
2. Con los pesos finales, comprueba a mano la entrada `[1, 1]`: ¿da `z > 0`? ¿Y `[1, 0]`?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. A partir de la **época 6** los errores son 0 y los pesos ya no cambian: **aprendió**.
2. Los pesos finales son `w = [0.2, 0.1]`, `b = −0.2`:
   - `[1, 1]`: 0.2 + 0.1 − 0.2 = **+0.1** > 0 → 1 ✅
   - `[1, 0]`: 0.2 − 0.2 = **0** → no es > 0 → 0 ✅

Nadie le dijo "las dos tienen que estar encendidas": lo **descubrió** a partir de 4 ejemplos.
</details>

### ✏️ Ejercicio 5 — Ahora tú: la puerta OR
La puerta **OR** vale 1 si **al menos una** entrada está encendida. Completa las etiquetas `Y_OR` (cambia los `?` por 0 o 1) y ejecuta.

In [ ]:
Y_OR = np.array([0, 1, 1, 1])     # ✏️ comprueba: [0,0]→?, [0,1]→?, [1,0]→?, [1,1]→?
w_or, b_or = entrenar_perceptron(X_logica, Y_OR)

<details><summary><b>👀 Ver respuesta</b></summary>

`Y_OR = [0, 1, 1, 1]`. El perceptrón la aprende aún más rápido: desde la **época 4** ya no comete errores, con pesos `[0.1, 0.1]` y sesgo `0`. Cualquier entrada encendida basta para que `z > 0`.
</details>

---
## 4. 📖 ¿Qué pasa si… le pedimos el XOR?

La puerta **XOR** ("o exclusivo") vale 1 cuando las entradas son **distintas**: `[0,1]` y `[1,0]` → 1; `[0,0]` y `[1,1]` → 0. Parece tan fácil como AND u OR.

### 🔮 Ejercicio 6 — Predice antes de ejecutar
Le damos al perceptrón **100 épocas** para aprender el XOR. ¿Llegará a 0 errores?

✍️ *Mi predicción:*

In [ ]:
Y_XOR = np.array([0, 1, 1, 0])
w_xor, b_xor = entrenar_perceptron(X_logica, Y_XOR, epocas=100, verbose=False)
pred = (X_logica @ w_xor + b_xor > 0).astype(int)
print("Correcto :", Y_XOR)
print("Predicho :", pred, f"  → aciertos: {(pred == Y_XOR).sum()} de 4")

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Tras 100 épocas sigue fallando: acierta como mucho **2 de 4** (con estos datos se queda dando vueltas y termina acertando 2), y los pesos dan vueltas sin parar. No es falta de paciencia: **es imposible**.
</details>

### 📖 ¿Por qué es imposible? Una neurona solo traza **una línea recta**

La neurona decide según el signo de `w₁·x₁ + w₂·x₂ + b`. La frontera entre "sí" y "no" (`z = 0`) es **una recta**. Mira los cuatro puntos:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, (nombre, Y, (w_, b_)) in zip(axes, [("AND", Y_AND, (w_and, b_and)),
                                            ("OR", Y_OR, (w_or, b_or)),
                                            ("XOR", Y_XOR, (w_xor, b_xor))]):
    ax.scatter(*X_logica.T, c=Y, cmap="coolwarm", s=250, edgecolor="k", vmin=0, vmax=1)
    xs = np.linspace(-0.5, 1.5, 10)
    if w_[1] != 0:
        ax.plot(xs, -(w_[0] * xs + b_) / w_[1], "k--", label="frontera aprendida")
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_title(nombre); ax.grid(alpha=.3)
plt.suptitle("rojo = 1 · azul = 0"); plt.tight_layout(); plt.show()

En AND y OR una recta separa los rojos de los azules. En XOR los rojos están en **esquinas opuestas**: no existe ninguna recta que deje los dos rojos de un lado y los dos azules del otro. Los datos que sí se pueden separar con una recta se llaman **linealmente separables**.

> 📜 **Historia:** en 1969, Minsky y Papert demostraron esta limitación y el entusiasmo por las redes neuronales se congeló durante años (el primer "invierno de la IA"). La solución, que verás en la **Lección 9.2**, es sorprendentemente simple: **usar varias neuronas en capas**.

### ✋ Ejercicio 7 — ¿Recta o no recta?
¿Cuáles de estos problemas puede resolver **una sola neurona**?
- a) Aprobar si `nota_examen + nota_tareas > 12`
- b) Alarma si la temperatura está **fuera** del rango 18–24 °C (ni muy frío ni muy caliente)
- c) Spam si el correo tiene muchas palabras como "gratis" y "oferta"

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **a) Sí.** Es literalmente una suma ponderada contra un umbral.
- **b) No.** La alarma suena en **dos zonas separadas** (abajo de 18 y arriba de 24) con la zona "normal" en medio. Una recta (aquí, un único corte en el eje de temperatura) no puede aislar el centro. Es un primo del XOR.
- **c) Sí (aproximadamente).** Cada palabra suma evidencia; es justo lo que hace una neurona (y lo que hace Naive Bayes, Lección 5.6).
</details>

---
## 5. 📖 La neurona moderna = regresión logística

La neurona de hoy no usa el escalón ni la regla del perceptrón. Usa:
- la **sigmoide** como activación (da probabilidades), y
- **descenso de gradiente** (Lección 3.3) sobre la **entropía cruzada** (Lección 3.2) para ajustar pesos y sesgo.

Si eso te suena familiar es porque **ya lo viste**: una neurona sigmoide **es exactamente una regresión logística** (Lección 5.2). Misma fórmula, mismo entrenamiento. La diferencia es de nombre y de lo que viene después: las redes **conectan muchas** de estas neuronas.

Vamos a entrenar una neurona en **PyTorch**. Un instituto quiere predecir si un estudiante **aprueba** a partir de sus horas de estudio y de sueño (datos ficticios, ya escalados como en la Lección 2.3).

In [ ]:
rng = np.random.default_rng(1)
n = 200
estudio = rng.normal(0, 1, n)                     # horas de estudio (escaladas)
sueno   = rng.normal(0, 1, n)                     # horas de sueño   (escaladas)
regla   = 2.0 * estudio + 1.0 * sueno - 0.5       # la regla oculta (el modelo NO la conoce)
aprueba = (regla + rng.normal(0, 0.7, n) > 0).astype(np.float32)

X = torch.tensor(np.c_[estudio, sueno], dtype=torch.float32)
y = torch.tensor(aprueba).unsqueeze(1)            # forma (200, 1)
print("Forma de X:", tuple(X.shape), "· aprueban:", f"{aprueba.mean():.0%}")

plt.scatter(estudio, sueno, c=aprueba, cmap="coolwarm", edgecolor="k", s=25)
plt.xlabel("estudio"); plt.ylabel("sueño"); plt.title("rojo = aprueba · azul = reprueba"); plt.show()

En PyTorch, una neurona es `nn.Linear(2, 1)` (2 entradas → 1 salida: hace la suma ponderada + sesgo) seguida de `nn.Sigmoid()`. El bucle de entrenamiento lo desmenuzaremos en la **Lección 9.4**; por ahora basta con saber que, en cada vuelta, **mide el error y mueve los pesos un poquito para reducirlo**.

### 🔮 Ejercicio 8 — Antes de entrenar
La regla oculta es `2·estudio + 1·sueño − 0.5`. Después de entrenar, ¿esperas que el peso del estudio sea **mayor**, **menor** o **igual** que el del sueño? ¿El sesgo será positivo o negativo?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
neurona = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())       # ¡una neurona!
perdida_fn = nn.BCELoss()                                     # entropía cruzada binaria (Lección 3.2)
optimizador = torch.optim.SGD(neurona.parameters(), lr=0.5)  # descenso de gradiente (Lección 3.3)

historia = []
for paso in range(300):
    prob = neurona(X)                  # 1) predecir
    perdida = perdida_fn(prob, y)      # 2) medir el error
    optimizador.zero_grad()
    perdida.backward()                 # 3) calcular hacia dónde mover cada peso (Lección 9.4)
    optimizador.step()                 # 4) mover los pesos un poquito
    historia.append(perdida.item())

capa = neurona[0]
print("Pesos aprendidos :", capa.weight.data.numpy().round(2))
print("Sesgo aprendido  :", capa.bias.data.numpy().round(2))
print(f"Exactitud        : {((neurona(X) > 0.5).float() == y).float().mean():.0%}")
plt.plot(historia); plt.xlabel("paso"); plt.ylabel("pérdida"); plt.title("La pérdida baja: la neurona aprende"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

El peso del estudio (**≈4.2**) sale aproximadamente el **doble** que el del sueño (**≈2.3**), igual que en la regla oculta, y el sesgo es **negativo (≈−1.1)**. Los números no son exactamente 2, 1 y −0.5 (los pesos salen "estirados" porque la neurona solo ve si aprobó o no, no el valor de la regla), pero lo importante, **la proporción y el signo**, lo descubrió sola. Exactitud ≈ 88 %.

Fíjate en la curva: la pérdida baja rápido al principio y luego se aplana. Es una **curva de aprendizaje** (Lección 3.7).
</details>

### ✋ Ejercicio 9 — ¿De verdad es una regresión logística?
Entrenamos una `LogisticRegression` de scikit-learn con los mismos datos (sin regularización, para que sea comparable). Compara sus coeficientes con los de la neurona.

In [ ]:
logistica = LogisticRegression(C=1e6).fit(X.numpy(), aprueba)    # C enorme = sin regularización
print("Regresión logística → pesos:", logistica.coef_.round(2), " sesgo:", logistica.intercept_.round(2))
print("Neurona PyTorch     → pesos:", capa.weight.data.numpy().round(2), " sesgo:", capa.bias.data.numpy().round(2))

✍️ *¿Qué observas?*

<details><summary><b>👀 Ver respuesta</b></summary>

Apuntan al **mismo lugar**: en los dos, el peso del estudio es ≈1.8 veces el del sueño y el sesgo es negativo (logística ≈ 5.3, 2.9 y −1.3; neurona ≈ 4.2, 2.3 y −1.1). No son idénticos porque la neurona solo dio 300 pasos de descenso de gradiente, mientras que scikit-learn usa un optimizador que llega hasta el final del camino; si le das más pasos a la neurona (✏️ cambia `range(300)` por `range(5000)`), sus números se acercan a los de la logística. **Misma fórmula, misma pérdida → misma solución.**
</details>

### 📖 La frontera de decisión: otra vez una recta
Como la neurona decide según `w₁·estudio + w₂·sueño + b > 0`, su frontera es una **recta**. ✏️ Prueba a cambiar `UMBRAL` a 0.9: ¿qué pasa con la recta?

In [ ]:
UMBRAL = 0.5     # ✏️ prueba 0.9 o 0.1

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
malla = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    prob_malla = neurona(malla).numpy().reshape(xx.shape)

plt.contourf(xx, yy, prob_malla, levels=20, cmap="coolwarm", alpha=.4)
plt.contour(xx, yy, prob_malla, levels=[UMBRAL], colors="k", linewidths=2)
plt.scatter(estudio, sueno, c=aprueba, cmap="coolwarm", edgecolor="k", s=20)
plt.xlabel("estudio"); plt.ylabel("sueño"); plt.title(f"Frontera (línea negra) con umbral {UMBRAL}"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con umbral 0.9 la recta se **desplaza** hacia la zona roja (hay que estudiar/dormir más para que la neurona diga "aprueba"), pero **sigue siendo una recta paralela**. Cambiar el umbral mueve la frontera; no puede curvarla. Para fronteras curvas hacen falta **más neuronas** (Lección 9.2).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Una neurona (o logística) no pasa de cierto acierto, aunque entrenes más | El problema **no es linealmente separable** (tipo XOR) | Añadir capas ocultas (Lección 9.2) o crear características nuevas |
| Un peso enorme domina todo y el entrenamiento es inestable | Entradas en escalas muy distintas (pesos en pesos, edad en años) | **Escalar** las entradas (Lección 2.3) |
| El perceptrón "baila" y nunca se estabiliza | Datos no separables: la regla del perceptrón no converge | Usar sigmoide + descenso de gradiente |
| Pesos que crecen sin parar con datos perfectamente separables | Sin ruido, la pérdida se reduce siempre "estirando" los pesos | Regularización (Lección 7.1, 9.8) o parar antes (7.2) |
| Interpretar el peso como "importancia" sin escalar | El peso depende de las unidades de la entrada | Comparar pesos solo con entradas escaladas |

### 🔮 Ejercicio 10 — Escalas mezcladas
Un compañero añade una tercera entrada: el **ingreso familiar en pesos** (valores alrededor de 20,000), sin escalar, junto a `estudio` y `sueño` (valores entre −3 y 3). Antes de entrenar, con pesos iniciales pequeños y parecidos, ¿qué entrada dominará la suma `z`? ¿Qué le pasará a la sigmoide?

✍️ *Mi predicción:*

In [ ]:
x_sin_escalar = np.array([0.5, -0.2, 20_000])     # estudio, sueño, ingreso en pesos
w_inicial = np.array([0.1, 0.1, 0.1])
z = np.dot(w_inicial, x_sin_escalar)
print(f"Aporte de cada entrada a z: {np.round(w_inicial * x_sin_escalar, 2)}")
print(f"z = {z:,.1f} → sigmoide(z) = {1 / (1 + np.exp(-z)):.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El ingreso aporta **2,000** a `z`; estudio y sueño aportan centésimas. La sigmoide queda **pegada en 1.0000**: la neurona ya "decidió" antes de aprender, y como la curva ahí es plana, los ajustes de pesos casi no cambian nada (verás por qué esto frena el aprendizaje en las Lecciones 9.3 y 9.7). **Escala siempre las entradas de una red.**
</details>

---
## 7. 📋 Ficha del modelo

| | Neurona artificial (perceptrón / neurona sigmoide) |
|---|---|
| **Tipo** | Supervisado: clasificación binaria (con otras activaciones, también regresión) |
| **Intuición** | Suma evidencias con pesos, añade un sesgo y decide con una activación |
| **Cómo aprende** | Perceptrón: corrige pesos solo cuando se equivoca · Neurona sigmoide: descenso de gradiente sobre la entropía cruzada |
| **Parámetros** | Un peso por entrada + un sesgo |
| **Hiperparámetros clave** | Tasa de aprendizaje (`lr`), número de pasos/épocas, activación |
| **✅ Fortalezas** | Simple, rápida, interpretable (con entradas escaladas) · Es el **ladrillo** de todas las redes |
| **❌ Debilidades** | Solo traza fronteras **rectas**: no resuelve XOR ni problemas "en zonas" |
| **Úsala cuando…** | Quieres entender las redes, o el problema es casi lineal (ahí es una logística) |
| **Evítala cuando…** | La relación es claramente no lineal: necesitas **capas** (9.2) |

## 8. 🧠 Resumen en 5 puntos
1. Una neurona hace dos cosas: **suma ponderada + sesgo**, y luego **activación** (decisión).
2. Los **pesos** dicen qué importa y en qué dirección; el **sesgo** dice cuánta evidencia hace falta.
3. El **perceptrón** aprende corrigiendo los pesos solo cuando se equivoca.
4. Una neurona sola traza **una recta**: no puede con el **XOR** ni con problemas no linealmente separables.
5. Una neurona con **sigmoide** entrenada con descenso de gradiente **es una regresión logística**; las redes nacen al **conectar muchas**.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Una neurona tiene pesos `[2, −1]`, sesgo `−1` y activación escalón (`1` si `z > 0`). ¿Qué devuelve para la entrada `[1, 1]`?
- a) 1 · b) 0 · c) 0.5 · d) −1

**2.** ¿Qué hace el **sesgo** en una neurona?
- a) Multiplica a todas las entradas
- b) Desplaza el umbral de decisión
- c) Elige qué entradas usar
- d) Evita el sobreajuste

**3.** ¿Cuál de estas puertas lógicas **no** puede aprender un único perceptrón?
- a) AND · b) OR · c) NOT · d) XOR

**4.** Una neurona con activación sigmoide entrenada con entropía cruzada es equivalente a…
- a) Un árbol de decisión · b) KNN · c) Una regresión logística · d) K-means

**5.** En la regla del perceptrón, si la neurona **acierta** un ejemplo, ¿cuánto cambian los pesos?
- a) Mucho · b) Un poquito · c) Nada · d) Se reinician a cero

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** `z = 2·1 + (−1)·1 − 1 = 0`, y 0 **no** es mayor que 0 → salida 0. ¡Ojo con los empates!
2. **b)** Es equivalente a mover el umbral: con sesgo más negativo, la neurona es "más difícil de convencer".
3. **d)** El XOR no es linealmente separable. (NOT sí: una sola entrada con peso negativo.)
4. **c)** Misma fórmula y misma pérdida (Lección 5.2).
5. **c)** El término `(correcto − predicho)` vale 0.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Tu cerebro tiene unos 86 mil millones de neuronas biológicas. ¿En qué se **parece** una neurona artificial a una biológica, y en qué **no**? (Pista: piensa en "sumar señales y dispararse", y luego en todo lo demás.)

**B. Caso de análisis 💼** Un hospital usa una sola neurona para decidir si un paciente necesita una prueba extra, con entradas `edad`, `presión`, `colesterol` (todas escaladas). El modelo aprendió los pesos `[0.1, 2.3, 0.9]`. El director dice: *"La presión es 23 veces más importante que la edad"*. ¿Estás de acuerdo? ¿Qué matices le darías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Se parece en la idea: recibe señales de otras neuronas, unas excitan y otras inhiben (pesos positivos y negativos), y "se dispara" si el total supera un umbral. **No** se parece en casi todo lo demás: las biológicas se comunican con pulsos en el tiempo, cambian químicamente, se conectan de formas muchísimo más complejas y no aprenden con descenso de gradiente. La neurona artificial es una **inspiración**, no una copia.

**B.** Con matices. Como las entradas están **escaladas**, comparar pesos sí es razonable: dentro del modelo, un cambio de una desviación estándar en presión mueve `z` unas 23 veces más que el mismo cambio en edad. Pero: (1) eso habla del **modelo**, no de la medicina; si edad y presión están correlacionadas, el modelo puede "repartir" el efecto de forma arbitraria; (2) un modelo lineal puede no captar interacciones (p. ej. presión alta es peor a mayor edad); (3) habría que mirar el acierto del modelo antes de interpretar nada.
</details>

### 9.3 🧩 Práctica integradora: "¿Recta o no recta?"

Un parque de diversiones mide, para cada visitante, la **altura** y la **edad** (escaladas), y quiere predecir si se **marea** en una atracción. Vas a comprobar si una sola neurona basta. 5 pasos cortos, cada uno con su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos.** Ejecuta y observa dónde están los que se marean.

In [ ]:
rng_p = np.random.default_rng(3)
n_p = 300
altura = rng_p.normal(0, 1, n_p)
edad = rng_p.normal(0, 1, n_p)
# Se marean los muy jóvenes y los muy mayores (ambos extremos de edad)
se_marea = (np.abs(edad) + rng_p.normal(0, 0.3, n_p) > 1.0).astype(np.float32)

X_p = torch.tensor(np.c_[altura, edad], dtype=torch.float32)
y_p = torch.tensor(se_marea).unsqueeze(1)
print(f"Se marea el {se_marea.mean():.0%} de los visitantes")
plt.scatter(altura, edad, c=se_marea, cmap="coolwarm", edgecolor="k", s=20)
plt.xlabel("altura"); plt.ylabel("edad"); plt.title("rojo = se marea · azul = no se marea"); plt.show()

✋ ¿Puedes trazar **una** recta que separe rojos de azules? ¿A qué ejemplo de la lección se parece?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No. Los rojos están **arriba y abajo** (edades extremas) y los azules en la franja del medio. Es el caso de la **alarma de temperatura** del Ejercicio 7: un patrón "en zonas", primo del XOR. La altura, además, parece no importar.
</details>

**Paso 2 🔮 — Línea base.** ¿Qué exactitud tendría un modelo "tonto" que **siempre** dice "no se marea"?

✍️ *Mi apuesta:*

In [ ]:
print(f"Exactitud de 'nadie se marea': {1 - se_marea.mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈ **72 %**, porque el 28 % se marea. Cualquier modelo debe ganarle a ese número (Lección 4.7).
</details>

**Paso 3 🔮 — Una neurona.** Entrenamos una neurona sigmoide con el mismo código de la Sección 5. ¿Le ganará claramente a la línea base?

✍️ *Mi apuesta:*

In [ ]:
def entrenar_neurona(X, y, pasos=500, lr=0.5):
    torch.manual_seed(0)
    modelo = nn.Sequential(nn.Linear(X.shape[1], 1), nn.Sigmoid())
    opt = torch.optim.SGD(modelo.parameters(), lr=lr)
    for _ in range(pasos):
        perdida = nn.BCELoss()(modelo(X), y)
        opt.zero_grad(); perdida.backward(); opt.step()
    exactitud = ((modelo(X) > 0.5).float() == y).float().mean().item()
    return modelo, exactitud

modelo_p, acc = entrenar_neurona(X_p, y_p)
print(f"Exactitud de una neurona: {acc:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈ **72 %**: **igual que la línea base**. La mejor recta que encuentra es, en la práctica, "decir casi siempre que no". No es un problema de entrenar más: es la limitación de la Sección 4.
</details>

**Paso 4 ✏️ — Ayuda a la neurona con una característica nueva.** Si la neurona solo traza rectas, podemos darle una entrada que haga el problema "recto". Añade como tercera entrada `edad**2` (la edad al cuadrado: grande en los dos extremos). Cambia `NUEVA` y ejecuta.

In [ ]:
NUEVA = edad ** 2          # ✏️ prueba también np.abs(edad), o altura ** 2 para comparar

X_p2 = torch.tensor(np.c_[altura, edad, NUEVA], dtype=torch.float32)
modelo_p2, acc2 = entrenar_neurona(X_p2, y_p)
print(f"Exactitud con la nueva característica: {acc2:.0%}")
print("Pesos [altura, edad, nueva]:", modelo_p2[0].weight.data.numpy().round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

Con `edad**2` la exactitud sube a ≈ **85 %**. El peso más grande es, con diferencia, el de la nueva característica: la neurona dice "si la edad al cuadrado es grande, te mareas". Con `altura ** 2` no mejora nada (se queda en ≈ 72 %); con `np.abs(edad)` funciona igual de bien (≈ 85 %): la característica tiene que capturar el patrón **real**.

Esto es **ingeniería de características** (Lección 2.4): tú le diste a la neurona la curva que no podía dibujar.
</details>

**Paso 5 ✋ — La gran pregunta.** Aquí nosotros adivinamos que `edad**2` ayudaría porque vimos la gráfica. ¿Qué harías con 100 entradas, o con los píxeles de una foto, donde no puedes "ver" qué característica inventar?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No podrías inventarlas a mano. La idea clave de las redes neuronales es **dejar que la red invente sus propias características**: una primera capa de neuronas crea características nuevas (como "edad extrema") y la siguiente neurona las combina. Eso es exactamente lo que verás en la **Lección 9.2: capas y profundidad**.
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *TensorFlow Playground* (playground.tensorflow.org): quita todas las capas ocultas y verás una sola neurona luchando con el dataset "XOR" o "círculo".
- 📺 **Video:** 3Blue1Brown, "¿Qué es una red neuronal?" (capítulo 1 de la serie *Deep Learning*, con subtítulos en español).
- 📘 **Lectura:** Michael Nielsen, *Neural Networks and Deep Learning*, capítulo 1 (perceptrones y neuronas sigmoide), gratis en línea.

---
⬅️ **Anterior:** [8.8 Proyecto del módulo: Los clientes del distribuidor 🚚](../../nivel-3-practicante/m08-no-supervisado/8.8-proyecto-del-modulo.md) · ➡️ **Siguiente:** [9.2 Capas y profundidad: por qué apilar funciona](9.2-capas-y-profundidad.ipynb)